## Scene 생성

## 0. Import

In [ ]:
import os

import sionna.rt

import trimesh

import uuid
from typing import Tuple

import drjit as dr
import mitsuba as mi
import numpy as np
import importlib
import matplotlib.pyplot as plt
import mitsuba as mi

print("Mitsuba variant :", mi.variant())

from matplotlib.patches import Rectangle, Circle
from sionna.rt import (
    load_scene,
    Transmitter,
    Receiver,
    PlanarArray,
    PathSolver,
    InteractionType,
    RadioMaterialBase,
    subcarrier_frequencies,
)
from sionna.rt.constants import NO_JONES_MATRIX
from sionna.rt.utils import jones_matrix_to_world_implicit, isclose

print("Sionna RT version:", sionna.rt.__version__)
print("Mitsuba variant  :", mi.variant())

## 0. project_config.py 불러오기

In [ ]:
import project_config

importlib.reload(project_config)

from project_config import (
    ROOM_X, ROOM_Y, ROOM_Z,
    BS_CENTER, RX_HEIGHT,
    FREQ_HZ, BW_HZ, N_SC, DF_HZ,
    NF_DB, P_REF_DBM, NOISE_SC_DBM,
    GRID_RES, N_COLS, N_ROWS,
    CELL_X, CELL_Y, RX_POINTS,
    SCENE_DIR, MESH_DIR, SCENE_XML,
    MATERIALS, OBSTACLES,
)

## 1. Obstacle을 얇은 평면으로 변환하는 함수

In [ ]:
# ------------------------------------
# obstacle을 얇은 평면으로 변환하는 함수
# ------------------------------------
def slab_geometry(ob):
    '''Return proxy width, thickness, height, and xy normal.'''
    if ob["kind"] == "cylinder":
        width = thickness = 2.0 * ob["radius"]
        height = ob["height"]
    else:
        dx, dy, height = ob["size"]
        if "proxy_normal" in ob:
            normal = np.asarray(ob["proxy_normal"], dtype=float)
            width, thickness = dy, dx
            return width, thickness, height, normal/np.linalg.norm(normal)
        # Cabinets are square in top view; use their side length as slab width/thickness.
        width, thickness = max(dx, dy), min(dx, dy)

    # Face the proxy toward the BS. This is exact for the direct BS bearing,
    # and an approximation for rays arriving after an exterior-wall reflection.
    normal = BS_CENTER[:2] - np.asarray(ob["center"], dtype=float)
    if np.linalg.norm(normal) < 1e-9:
        normal = np.array([1.0, 0.0])
    normal = normal / np.linalg.norm(normal)
    return width, thickness, height, normal

## 2. 사용자 정의 재질: project_radio_material.py 불러오기

In [ ]:
from project_radio_material import (
    ProjectRadioMaterial,
    PLUGIN_NAME,
)

print("Registered Mitsuba BSDF plugin:", PLUGIN_NAME)

## 3. 전체 scene의 mesh와 XML 생성

In [ ]:
MESH_DIR.mkdir(parents=True, exist_ok=True)

# 수직면 만드는 함수
def make_vertical_plane(center_xy, width, height, normal_xy):
    '''Two triangles forming one vertical rectangular interaction surface.'''
    center = np.asarray(center_xy, dtype=float)
    normal = np.asarray(normal_xy, dtype=float)
    normal = normal / np.linalg.norm(normal)
    tangent = np.array([-normal[1], normal[0]])
    p0 = center - 0.5*width*tangent
    p1 = center + 0.5*width*tangent
    vertices = np.array([
        [p0[0], p0[1], 0.0],
        [p1[0], p1[1], 0.0],
        [p1[0], p1[1], height],
        [p0[0], p0[1], height],
    ])
    faces = np.array([[0, 1, 2], [0, 2, 3]], dtype=np.int64)
    return trimesh.Trimesh(vertices=vertices, faces=faces, process=False)


# 외벽 4개
def perimeter_objects():
    return [
        dict(name="ROOM_Wall_West", center=(0.0, ROOM_Y/2),
             width=ROOM_Y, height=ROOM_Z, normal=(1.0, 0.0)),
        dict(name="ROOM_Wall_East", center=(ROOM_X, ROOM_Y/2),
             width=ROOM_Y, height=ROOM_Z, normal=(-1.0, 0.0)),
        dict(name="ROOM_Wall_South", center=(ROOM_X/2, 0.0),
             width=ROOM_X, height=ROOM_Z, normal=(0.0, 1.0)),
        dict(name="ROOM_Wall_North", center=(ROOM_X/2, ROOM_Y),
             width=ROOM_X, height=ROOM_Z, normal=(0.0, -1.0)),
    ]


# material 이름 생성
def material_key(material_name, thickness):
    return f"{material_name}_{thickness:.3f}".replace(".", "p")

entries = []
material_specs = {}


# Perimeter reflection planes (외벽 재질 설정)
perim_key = "perimeter_reflector"
material_specs[perim_key] = dict(
    mode=ProjectRadioMaterial.REFLECT,
    rho=MATERIALS["perimeter"]["rho"],
    alpha_db_per_m=0.0,
    thickness=0.1,
    color=MATERIALS["perimeter"]["color"],
)
for ob in perimeter_objects():
    mesh = make_vertical_plane(ob["center"], ob["width"],
                               ob["height"], ob["normal"])
    filename = f"{ob['name']}.ply"
    mesh.export(os.path.join(MESH_DIR, filename))
    entries.append(dict(name=ob["name"], filename=filename,
                        material_key=perim_key, role="perimeter"))


# Obstacle 재질 설정
for ob in OBSTACLES:
    width, thickness, height, normal = slab_geometry(ob)
    key = material_key(ob["material"], thickness)
    mat = MATERIALS[ob["material"]]
    material_specs[key] = dict(
        mode=ProjectRadioMaterial.TRANSMIT,
        rho=0.0,
        alpha_db_per_m=mat["alpha_db_per_m"],
        thickness=thickness,
        color=mat["color"],
    )
    mesh = make_vertical_plane(ob["center"], width, height, normal)
    filename = f"{ob['name']}.ply"
    mesh.export(os.path.join(MESH_DIR, filename))
    entries.append(dict(name=ob["name"], filename=filename,
                        material_key=key, role="interior"))


# 색상을 XML 문자열로 변환
def rgb_text(rgb):
    return " ".join(f"{x:.4f}" for x in rgb)


# XML 파일 생성
def build_scene_xml(path):
    lines = ['<scene version="3.0.0">', '  <integrator type="path"/>']
    # 사용자 정의 material 기록
    for key, spec in material_specs.items():
        lines += [
            f'  <bsdf type="{PLUGIN_NAME}" id="mat-{key}">',
            f'    <integer name="mode" value="{spec["mode"]}"/>',
            f'    <float name="rho" value="{spec["rho"]}"/>',
            f'    <float name="alpha_db_per_m" value="{spec["alpha_db_per_m"]}"/>',
            f'    <float name="thickness" value="{spec["thickness"]}"/>',
            f'    <rgb name="color" value="{rgb_text(spec["color"])}"/>',
            '  </bsdf>',
        ]
    # 물체와 material 연결
    for e in entries:
        lines += [
            f'  <shape type="ply" id="mesh-{e["name"]}">',
            f'    <string name="filename" value="meshes/{e["filename"]}"/>',
            f'    <ref name="bsdf" id="mat-{e["material_key"]}"/>',
            '  </shape>',
        ]
    lines.append('</scene>')
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))


# XML 파일 저장
build_scene_xml(SCENE_XML)

print("Scene XML :", SCENE_XML)
print("Objects   :", len(entries), "= 4 perimeter +", len(OBSTACLES), "interior")
print("생성된 Materials :", list(material_specs))

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch


# =========================================================
# 3D object 그리기 함수
# =========================================================
def add_box_3d(ax, center, size, color, alpha=0.82):
    cx, cy = center
    dx, dy, dz = size

    ax.bar3d(
        cx - dx/2,
        cy - dy/2,
        0,
        dx, dy, dz,
        color=color,
        edgecolor="0.25",
        linewidth=0.45,
        alpha=alpha,
        shade=True,
    )


def add_cylinder_3d(ax, center, radius, height, color, alpha=0.86):
    theta = np.linspace(0, 2*np.pi, 36)
    z = np.array([0.0, height])
    theta, z = np.meshgrid(theta, z)

    x = center[0] + radius*np.cos(theta)
    y = center[1] + radius*np.sin(theta)

    ax.plot_surface(
        x, y, z,
        color=color,
        edgecolor="none",
        alpha=alpha,
        shade=True,
    )


# =========================================================
# 3D scene 생성
# =========================================================
fig = plt.figure(figsize=(12, 7.2), facecolor="white")
ax = fig.add_subplot(111, projection="3d")


# ---------------------------------------------------------
# 바닥
# ---------------------------------------------------------
xx, yy = np.meshgrid(
    [0, ROOM_X],
    [0, ROOM_Y],
)

ax.plot_surface(
    xx,
    yy,
    np.zeros_like(xx),
    color="#dfe5eb",
    alpha=0.45,
    shade=False,
)


# ---------------------------------------------------------
# 방 전체 경계
# ---------------------------------------------------------
corners = np.array([
    [0,      0,      0],
    [ROOM_X, 0,      0],
    [ROOM_X, ROOM_Y, 0],
    [0,      ROOM_Y, 0],
    [0,      0,      ROOM_Z],
    [ROOM_X, 0,      ROOM_Z],
    [ROOM_X, ROOM_Y, ROOM_Z],
    [0,      ROOM_Y, ROOM_Z],
])

edges = [
    (0,1), (1,2), (2,3), (3,0),
    (4,5), (5,6), (6,7), (7,4),
    (0,4), (1,5), (2,6), (3,7),
]

for a, b in edges:
    ax.plot(
        *zip(corners[a], corners[b]),
        color="#77818c",
        linewidth=0.8,
        alpha=0.45,
    )


# ---------------------------------------------------------
# 내부 장애물
# ---------------------------------------------------------
for ob in OBSTACLES:
    material_color = MATERIALS[ob["material"]]["color"]

    if ob["kind"] == "cylinder":
        add_cylinder_3d(
            ax,
            center=ob["center"],
            radius=ob["radius"],
            height=ob["height"],
            color=material_color,
        )

    else:
        add_box_3d(
            ax,
            center=ob["center"],
            size=ob["size"],
            color=material_color,
        )


# ---------------------------------------------------------
# Base Station
# ---------------------------------------------------------
ax.scatter(
    *BS_CENTER,
    s=190,
    marker="*",
    color="#d62728",
    edgecolor="white",
    linewidth=0.8,
    depthshade=False,
)

ax.text(
    BS_CENTER[0],
    BS_CENTER[1],
    BS_CENTER[2] + 0.18,
    "BS",
    color="#a4161a",
    weight="bold",
)


# ---------------------------------------------------------
# AMR 출발점과 목표점
# ---------------------------------------------------------
amr_points = {
    "A1 start": (1.0,  2.0, 0.12, "#1f77b4", "o"),
    "A1 goal":  (17.0, 8.0, 0.12, "#1f77b4", "X"),
    "A2 start": (17.0, 2.0, 0.12, "#ff7f0e", "o"),
    "A2 goal":  (1.0,  8.0, 0.12, "#ff7f0e", "X"),
}

for label, (x, y, z, color, marker) in amr_points.items():
    ax.scatter(
        x, y, z,
        s=85,
        marker=marker,
        color=color,
        edgecolor="white",
        linewidth=0.8,
        depthshade=False,
    )

    ax.text(
        x, y, z + 0.15,
        label,
        color=color,
        fontsize=8,
        weight="bold",
    )


# =========================================================
# 그래프 설정
# =========================================================
ax.set_xlim(0, ROOM_X)
ax.set_ylim(0, ROOM_Y)
ax.set_zlim(0, ROOM_Z + 0.5)

ax.set_box_aspect((ROOM_X, ROOM_Y, ROOM_Z))

# 조감도 각도
ax.view_init(
    elev=27,
    azim=-58,
)

ax.set_xlabel("x [m]", labelpad=8)
ax.set_ylabel("y [m]", labelpad=8)
ax.set_zlabel("z [m]", labelpad=4)

ax.set_title(
    "3D Indoor Scene for Two-AMR Navigation",
    pad=16,
    weight="bold",
)

ax.grid(False)

for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
    axis.pane.set_alpha(0.0)


# =========================================================
# 범례
# =========================================================
legend_handles = [
    Patch(
        facecolor=MATERIALS["concrete"]["color"],
        label="Concrete",
    ),
    Patch(
        facecolor=MATERIALS["wood"]["color"],
        label="Wood",
    ),
    Line2D(
        [0], [0],
        marker="*",
        color="white",
        markerfacecolor="#d62728",
        markersize=13,
        label="Base station",
    ),
    Line2D(
        [0], [0],
        marker="o",
        color="white",
        markerfacecolor="#1f77b4",
        markersize=8,
        label="AMR start",
    ),
    Line2D(
        [0], [0],
        marker="X",
        color="white",
        markerfacecolor="#1f77b4",
        markersize=8,
        label="AMR goal",
    ),
]

ax.legend(
    handles=legend_handles,
    loc="upper left",
    bbox_to_anchor=(0.02, 0.98),
    frameon=False,
)

plt.tight_layout()